# Khối B — Huấn luyện YOLOv8s phát hiện + phân lớp vùng chữ (6 lớp)

Tái lập đúng lần train cho `code/weights/yolo_vaipe.pt` (plan.md §5.2, thí nghiệm E-B2).
Notebook **tự chứa**: mọi code nằm trong các cell dưới, không clone repo nào. Bước sinh nhãn cùng logic với
`code/tools/to_yolo.py` (đã đối chiếu: số box từng lớp và `split_manifest.csv` khớp 100 %).

| Tham số | Giá trị | Lý do |
| :--- | :--- | :--- |
| Mô hình | `yolov8s.pt` (COCO pretrained), `ultralytics==8.4.209` | |
| `imgsz` | 1024 | chữ VAIPE cao 15–25 px, 640 làm mất vùng nhỏ |
| `epochs` / `patience` | 50 / 15 | dừng sớm theo mAP val |
| `batch` | 16, DDP trên 2 × T4 (`device=0,1`) | |
| `fliplr` / `mosaic` | 0 / 0,5 | lật ngang làm hỏng chữ |
| `seed`, `deterministic` | 42, True | |

Kết quả lần chạy gốc (test, 176 ảnh, `best.pt` chọn theo val): **P 0,9947 · R 0,9957 · F1 0,9952 · mAP50 0,9947 · mAP50-95 0,9270**.
Thời gian ≈ 1,5–2,5 giờ.

## Cách chạy trên Kaggle

1. **Create → New Notebook → File → Import Notebook**, chọn file này.
2. **Add Input** → tìm `vaipepill2022` (của `tommyngx`) → **Add**. Không cần upload dữ liệu: notebook đọc thẳng
   `public_train/prescription/{image,label}` và chia split theo danh sách nhúng sẵn ở cell *Dữ liệu*.
3. **Session options**: Accelerator **GPU T4 x2**, Internet **On** (để `pip install` và tải trọng số pretrained).
4. Chạy thử: đặt `QUICK = True` ở cell *Tham số* → **Run All** (~10–15 phút) để bắt lỗi môi trường.
5. Chạy thật: `QUICK = False` → **Save Version → Save & Run All (Commit)**, *Advanced*: *Always save output*.
6. Lấy kết quả ở tab **Output**: `output/results.zip`. Đặt trọng số vào `code/weights/` của repo (tên ở cuối notebook).

Môi trường lúc chạy gốc (11/11/2026): image Kaggle Python 3.13.16, torch 2.11.0+cu130, 2 × Tesla T4.
Huấn luyện trên GPU không tất định hoàn toàn, nên chạy lại có thể lệch nhẹ ở chữ số thập phân thứ 3–4.

## Tham số

In [ ]:
QUICK = False            # True: chạy thử 1 epoch để kiểm tra môi trường
EPOCHS = 1 if QUICK else 50
IMGSZ, BATCH, PATIENCE, SEED = 1024, 16, 15, 42
DEVICE = '0,1'           # 2 × T4, DDP
WORK = '/kaggle/working'
OUT_DIR = f'{WORK}/output'

## Cài đặt

In [ ]:
!pip install -q "ultralytics==8.4.209"
import os
os.environ['WANDB_MODE'] = 'disabled'   # Kaggle có sẵn wandb, tránh hỏi đăng nhập
import sys, torch, ultralytics
print('python', sys.version.split()[0], '| torch', torch.__version__, '| ultralytics', ultralytics.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, 'Chưa bật GPU (Accelerator = GPU T4 x2)'

## Dữ liệu

In [ ]:
# ---- Dữ liệu: tìm public_train của vaipepill2022, chia split theo danh sách cố định ----
# Bộ chia = data/1_vaipe/split_manifest.csv của repo (gom theo bệnh nhân, seed 42): train/valid/test = 820/176/176.
# Mọi ảnh là VAIPE_P_TRAIN_<số>; VAIPE_P_TRAIN_1172 bị loại (trùng byte với VAIPE_P_TEST_32).
import collections, json, os, shutil, sys, unicodedata
from pathlib import Path

SPLIT_IDS = {"train": [0, 3, 4, 6, 7, 8, 9, 12, 13, 14, 16, 18, 22, 23, 24, 25, 26, 30, 31, 32, 34, 35, 37, 38, 39, 40, 41, 42, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 58, 59, 60, 61, 62, 63, 66, 67, 68, 69, 70, 72, 73, 75, 76, 77, 79, 80, 81, 83, 84, 85, 86, 87, 90, 91, 92, 95, 96, 98, 99, 101, 102, 103, 104, 107, 108, 109, 110, 111, 112, 114, 115, 116, 118, 119, 120, 121, 123, 124, 125, 126, 127, 128, 130, 131, 132, 135, 136, 138, 140, 141, 142, 143, 145, 147, 149, 152, 154, 159, 160, 162, 163, 166, 167, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 183, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 206, 208, 209, 211, 212, 213, 214, 218, 220, 221, 225, 226, 230, 235, 237, 238, 239, 240, 243, 244, 245, 246, 247, 249, 250, 251, 253, 254, 255, 256, 257, 259, 260, 263, 266, 267, 268, 270, 271, 272, 273, 274, 275, 276, 292, 294, 295, 296, 297, 300, 302, 303, 304, 305, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 325, 326, 327, 328, 329, 331, 333, 335, 336, 337, 338, 339, 341, 343, 344, 345, 347, 348, 349, 350, 351, 352, 354, 355, 356, 358, 359, 363, 365, 367, 368, 369, 371, 374, 375, 377, 379, 380, 382, 384, 385, 386, 387, 388, 391, 392, 394, 395, 396, 400, 401, 407, 408, 409, 410, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 430, 432, 433, 435, 436, 437, 440, 443, 444, 445, 447, 449, 451, 452, 455, 456, 457, 458, 463, 465, 466, 467, 468, 469, 472, 473, 474, 475, 476, 477, 479, 482, 483, 486, 489, 491, 492, 493, 494, 495, 496, 497, 498, 500, 503, 504, 505, 506, 508, 510, 512, 513, 514, 515, 516, 518, 519, 521, 523, 524, 525, 526, 527, 528, 529, 532, 533, 536, 538, 539, 540, 541, 542, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 559, 560, 561, 562, 563, 564, 565, 566, 568, 569, 570, 571, 573, 576, 577, 579, 581, 582, 584, 587, 588, 589, 590, 591, 592, 595, 596, 598, 599, 600, 601, 603, 604, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 620, 622, 623, 625, 627, 628, 629, 631, 633, 635, 636, 637, 638, 639, 640, 642, 643, 644, 647, 648, 649, 652, 653, 656, 657, 659, 660, 661, 662, 663, 664, 665, 667, 668, 669, 670, 671, 672, 673, 674, 676, 677, 678, 681, 682, 683, 685, 686, 688, 689, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703, 704, 706, 707, 708, 709, 710, 713, 716, 717, 718, 719, 721, 723, 724, 725, 726, 727, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 760, 761, 762, 763, 764, 765, 768, 770, 771, 774, 776, 777, 778, 779, 781, 782, 783, 785, 786, 787, 790, 791, 794, 795, 796, 799, 800, 801, 803, 804, 805, 806, 807, 808, 810, 811, 812, 813, 814, 815, 816, 817, 819, 820, 821, 822, 824, 825, 826, 827, 829, 830, 831, 832, 833, 836, 838, 839, 842, 844, 845, 846, 847, 848, 849, 850, 852, 854, 856, 857, 858, 859, 860, 861, 864, 865, 866, 867, 869, 870, 871, 872, 873, 874, 875, 877, 878, 879, 882, 883, 884, 885, 886, 887, 890, 891, 893, 895, 897, 898, 899, 900, 901, 903, 905, 906, 909, 910, 914, 916, 917, 918, 919, 920, 921, 922, 924, 925, 926, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 941, 942, 943, 944, 945, 946, 948, 949, 950, 951, 952, 953, 957, 958, 959, 962, 964, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 981, 982, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 996, 997, 998, 999, 1000, 1001, 1003, 1006, 1007, 1009, 1011, 1012, 1013, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1024, 1025, 1026, 1028, 1029, 1031, 1033, 1034, 1036, 1037, 1038, 1039, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1050, 1051, 1053, 1054, 1056, 1057, 1058, 1059, 1060, 1061, 1063, 1064, 1065, 1066, 1067, 1070, 1071, 1072, 1073, 1076, 1077, 1078, 1079, 1080, 1081, 1083, 1084, 1086, 1087, 1089, 1090, 1091, 1092, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1107, 1111, 1112, 1113, 1114, 1116, 1118, 1119, 1120, 1121, 1122, 1123, 1124, 1125, 1126, 1127, 1129, 1131, 1133, 1134, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1146, 1147, 1148, 1149, 1150, 1152, 1153, 1155, 1156, 1157, 1158, 1159, 1160, 1162, 1164, 1165, 1166, 1167, 1168, 1170, 1171], "valid": [2, 10, 11, 17, 27, 28, 44, 64, 65, 74, 88, 100, 106, 113, 122, 133, 134, 144, 146, 148, 150, 151, 153, 155, 156, 157, 158, 161, 164, 165, 168, 169, 200, 202, 210, 215, 219, 224, 229, 231, 233, 234, 236, 241, 248, 252, 258, 261, 265, 277, 280, 298, 299, 301, 306, 307, 308, 321, 322, 332, 340, 342, 346, 360, 361, 362, 364, 373, 376, 381, 389, 393, 399, 406, 429, 431, 434, 438, 439, 459, 462, 471, 478, 480, 484, 501, 507, 531, 535, 537, 543, 544, 558, 580, 585, 586, 593, 621, 624, 630, 641, 654, 655, 658, 675, 680, 684, 687, 705, 711, 714, 715, 720, 730, 759, 775, 780, 788, 793, 797, 798, 802, 823, 828, 834, 835, 837, 840, 841, 851, 862, 889, 892, 894, 902, 907, 912, 913, 915, 927, 928, 929, 940, 954, 955, 956, 960, 961, 963, 983, 984, 1002, 1004, 1008, 1010, 1014, 1023, 1040, 1052, 1062, 1068, 1069, 1074, 1088, 1093, 1094, 1105, 1106, 1108, 1110, 1115, 1117, 1130, 1145, 1151, 1169], "test": [1, 5, 15, 19, 20, 21, 29, 33, 36, 43, 56, 57, 71, 78, 82, 89, 93, 94, 97, 105, 117, 129, 137, 139, 180, 182, 189, 203, 204, 205, 207, 216, 217, 222, 223, 227, 228, 232, 242, 262, 264, 269, 278, 279, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 293, 323, 324, 330, 334, 353, 357, 366, 370, 372, 378, 383, 390, 397, 398, 402, 403, 404, 405, 411, 441, 442, 446, 448, 450, 453, 454, 460, 461, 464, 470, 481, 485, 487, 488, 490, 499, 502, 509, 511, 517, 520, 522, 530, 534, 567, 572, 574, 575, 578, 583, 594, 597, 602, 605, 606, 607, 619, 626, 632, 634, 645, 646, 650, 651, 666, 679, 690, 691, 692, 712, 722, 728, 729, 748, 766, 767, 769, 772, 773, 784, 789, 792, 809, 818, 843, 853, 855, 863, 868, 876, 880, 881, 888, 896, 904, 908, 911, 923, 947, 965, 980, 995, 1005, 1027, 1030, 1032, 1035, 1048, 1049, 1055, 1075, 1082, 1085, 1109, 1128, 1132, 1135, 1154, 1161, 1163]}
SPLITS = {s: sorted(f'VAIPE_P_TRAIN_{i}' for i in ids) for s, ids in SPLIT_IDS.items()}
CLASSES = ['drugname', 'quantity', 'usage', 'diagnose', 'date', 'other']   # thứ tự = class_id YOLO
EXPECTED = {  # số vùng chữ theo lớp — phải khớp, nếu không là sai dataset/split
    'train': {'images': 820, 'drugname': 1995, 'quantity': 1995, 'usage': 2809, 'diagnose': 1109, 'date': 820, 'other': 20108},
    'valid': {'images': 176, 'drugname': 405, 'quantity': 405, 'usage': 583, 'diagnose': 287, 'date': 176, 'other': 4402},
    'test':  {'images': 176, 'drugname': 443, 'quantity': 443, 'usage': 602, 'diagnose': 253, 'date': 176, 'other': 4169},
}


def find_public_train(search='/kaggle/input', max_depth=6):
    queue = [(Path(search), 0)]
    while queue:
        d, depth = queue.pop(0)
        if (d / 'prescription' / 'label').is_dir() and d.name == 'public_train':
            return d
        if depth < max_depth:
            try:
                queue += [(p, depth + 1) for p in sorted(d.iterdir()) if p.is_dir()
                          and p.name not in ('image', 'label', 'pill')]
            except OSError:
                pass
    return None


DATA_ROOT = Path(os.environ.get('DATA_ROOT') or find_public_train() or '')
LABEL_DIR, IMAGE_DIR = DATA_ROOT / 'prescription' / 'label', DATA_ROOT / 'prescription' / 'image'
assert LABEL_DIR.is_dir(), 'Không thấy public_train/prescription/label — Add Input dataset tommyngx/vaipepill2022'
on_disk = {p.stem for p in LABEL_DIR.glob('*.json')}
listed = {s for v in SPLITS.values() for s in v}
assert listed <= on_disk, f'Thiếu nhãn: {sorted(listed - on_disk)[:5]}'
print('DATA_ROOT =', DATA_ROOT)
print('nhãn không thuộc split nào (bỏ qua):', sorted(on_disk - listed))


def image_path(stem):
    for ext in ('.png', '.jpg', '.jpeg'):
        if (IMAGE_DIR / (stem + ext)).exists():
            return IMAGE_DIR / (stem + ext)
    raise FileNotFoundError(stem)


def read_label(stem):
    with open(LABEL_DIR / f'{stem}.json', encoding='utf-8') as f:
        return json.load(f)


def clean_text(text):
    """NFC + gộp khoảng trắng (= code/src/vaipe/dataset.py::clean_text)."""
    return ' '.join(unicodedata.normalize('NFC', str(text)).split())


def clip_box(box, w, h):
    """xyxy kẹp vào ảnh, làm tròn int (= code/src/common/boxes.py::clip_box)."""
    x1, y1, x2, y2 = (float(v) for v in box)
    x1, x2 = sorted((min(max(x1, 0), w), min(max(x2, 0), w)))
    y1, y2 = sorted((min(max(y1, 0), h), min(max(y2, 0), h)))
    return int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))


for split, stems in SPLITS.items():
    cnt = collections.Counter(a['label'] for s in stems for a in read_label(s))
    got = {'images': len(stems), **{c: cnt[c] for c in CLASSES}}
    print(f'{split:5s}:', got)
    assert got == EXPECTED[split], f'{split} lệch: {got} != {EXPECTED[split]}'

## Sinh bộ dữ liệu YOLO

Mỗi box → `class cx cy w h` chuẩn hóa; box kẹp vào ảnh, bỏ box ≤ 1 px. Ảnh được symlink, không copy.
Tên thư mục split theo ultralytics: `train / val / test` (`val` = `valid` của repo).

In [ ]:
from PIL import Image

YOLO_DIR = Path(WORK) / 'dataset' / 'yolov8'
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
cls_id = {c: i for i, c in enumerate(CLASSES)}
stats = {}
for split, folder in (('train', 'train'), ('valid', 'val'), ('test', 'test')):
    (YOLO_DIR / 'images' / folder).mkdir(parents=True)
    (YOLO_DIR / 'labels' / folder).mkdir(parents=True)
    per_class, n_skip = collections.Counter(), 0
    for stem in SPLITS[split]:
        img = image_path(stem)
        with Image.open(img) as im:
            w, h = im.size
        lines = []
        for a in read_label(stem):
            if a.get('label') not in cls_id:
                continue
            x1, y1, x2, y2 = clip_box(a['box'], w, h)
            if x2 - x1 <= 1 or y2 - y1 <= 1:
                n_skip += 1
                continue
            lines.append(f"{cls_id[a['label']]} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} "
                         f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
            per_class[a['label']] += 1
        os.symlink(img, YOLO_DIR / 'images' / folder / img.name)
        (YOLO_DIR / 'labels' / folder / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
    stats[folder] = {'images': len(SPLITS[split]), 'boxes': sum(per_class.values()),
                     'boxes_per_class': {c: per_class[c] for c in CLASSES}, 'skipped_degenerate_boxes': n_skip}
    print(folder, stats[folder])
    assert {c: per_class[c] for c in CLASSES} == {c: EXPECTED[split][c] for c in CLASSES} and n_skip == 0

(YOLO_DIR / 'data.yaml').write_text(
    f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n'
    + ''.join(f'  {i}: {c}\n' for i, c in enumerate(CLASSES)))
json.dump(stats, open(YOLO_DIR / 'stats.json', 'w'), indent=1)
print(open(YOLO_DIR / 'data.yaml').read())

## Huấn luyện + chấm điểm

Chạy dưới dạng script (không chạy trong kernel) vì DDP 2 GPU của ultralytics cần một tiến trình riêng — giống lần chạy gốc.
Sau khi train: chấm `best.pt` trên val (chọn ngưỡng `conf` tối đa F1 trung bình) rồi trên test.

In [ ]:
%%writefile /kaggle/working/train_yolo.py
"""YOLOv8 phát hiện + phân lớp vùng chữ đơn thuốc, 6 lớp (plan.md §5.2, khối B).
train → khớp mô hình; val → chọn best.pt, dừng sớm, ngưỡng conf tối đa F1; test → chấm 1 lần."""
import argparse
import json
from pathlib import Path

import yaml
from ultralytics import YOLO


def summarise(metrics, names):
    box = metrics.box
    p, r = float(box.mp), float(box.mr)
    out = {'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p + r else 0.0,
           'mAP50': float(box.map50), 'mAP50-95': float(box.map), 'per_class': {}}
    for i, c in enumerate(box.ap_class_index):
        cp, cr, ap50, ap = (float(v) for v in box.class_result(i))
        out['per_class'][names[int(c)]] = {
            'precision': cp, 'recall': cr, 'f1': 2 * cp * cr / (cp + cr) if cp + cr else 0.0,
            'mAP50': ap50, 'mAP50-95': ap}
    return out


def best_conf(metrics):
    try:
        f1 = metrics.box.f1_curve.mean(0)
        i = int(f1.argmax())
        return float(metrics.box.px[i]), float(f1[i])
    except Exception:
        return None, None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True)
    ap.add_argument('--model', default='yolov8s.pt')
    ap.add_argument('--epochs', type=int, default=50)
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--imgsz', type=int, default=1024)
    ap.add_argument('--device', default='0,1')
    ap.add_argument('--workers', type=int, default=4)
    ap.add_argument('--patience', type=int, default=15)
    ap.add_argument('--fliplr', type=float, default=0.0)   # lật ngang làm hỏng chữ
    ap.add_argument('--mosaic', type=float, default=0.5)
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--fraction', type=float, default=1.0)
    ap.add_argument('--project', default='/kaggle/working/runs')
    ap.add_argument('--name', default='vaipe6')
    opt = ap.parse_args()

    data_yaml = Path(opt.data).resolve()
    names = yaml.safe_load(open(data_yaml))['names']
    model = YOLO(opt.model)
    model.train(data=str(data_yaml), epochs=opt.epochs, batch=opt.batch, imgsz=opt.imgsz,
                device=opt.device, workers=opt.workers, patience=opt.patience,
                fliplr=opt.fliplr, mosaic=opt.mosaic, fraction=opt.fraction,
                seed=opt.seed, deterministic=True,
                project=opt.project, name=opt.name, exist_ok=True, plots=True)

    run_dir = Path(opt.project) / opt.name
    best = YOLO(str(run_dir / 'weights' / 'best.pt'))
    kw = dict(data=str(data_yaml), imgsz=opt.imgsz, batch=opt.batch, device=opt.device.split(',')[0],
              project=opt.project, exist_ok=True, plots=True)
    val = best.val(split='val', name=opt.name + '_val', **kw)
    conf, f1 = best_conf(val)
    test = best.val(split='test', name=opt.name + '_test', **kw)
    result = {'split': 'test', 'weights': 'best.pt (selected on val)', **summarise(test, names),
              'val': summarise(val, names), 'conf_max_f1_on_val': conf,
              'mean_f1_at_that_conf_on_val': f1,
              'params': {'model': opt.model, 'imgsz': opt.imgsz, 'epochs': opt.epochs,
                         'batch': opt.batch, 'patience': opt.patience, 'fliplr': opt.fliplr,
                         'mosaic': opt.mosaic, 'seed': opt.seed, 'fraction': opt.fraction}}
    json.dump(result, open(run_dir / 'test_metrics.json', 'w'), indent=1)
    print(f'conf that maximises mean F1 on val: {conf}')
    for c, m in [('all', result)] + list(result['per_class'].items()):
        print(f'{c:10s} P={m["precision"]:.4f} R={m["recall"]:.4f} F1={m["f1"]:.4f} '
              f'mAP50={m["mAP50"]:.4f} mAP50-95={m["mAP50-95"]:.4f}')
    print(f'TEST RESULT (YOLOv8): P={result["precision"]:.4f}  R={result["recall"]:.4f}  '
          f'F1={result["f1"]:.4f}  mAP50={result["mAP50"]:.4f}  mAP50-95={result["mAP50-95"]:.4f}')


if __name__ == '__main__':
    main()

In [ ]:
!mkdir -p {OUT_DIR}/logs
!python {WORK}/train_yolo.py --data {YOLO_DIR}/data.yaml --model yolov8s.pt --imgsz {IMGSZ} \
    --epochs {EPOCHS} --patience {PATIENCE} --batch {BATCH} --device {DEVICE} --workers 4 --seed {SEED} \
    2>&1 | tee {OUT_DIR}/logs/yolov8.log | grep -E "^ +all |Epoch|TEST RESULT|Error|Traceback" | tail -n 120

## Gom kết quả

In [ ]:
RUN = Path(WORK) / 'runs' / 'vaipe6'
dst = Path(OUT_DIR) / 'yolov8'
(dst / 'val').mkdir(parents=True, exist_ok=True)
(dst / 'test').mkdir(parents=True, exist_ok=True)
shutil.copy(RUN / 'weights' / 'best.pt', dst / 'yolo_vaipe.pt')
for pat in ('*.png', '*.csv', '*.yaml', 'test_metrics.json'):
    for f in RUN.glob(pat):
        shutil.copy(f, dst)
for sub in ('val', 'test'):
    for f in (RUN.parent / f'vaipe6_{sub}').glob('*.png'):
        shutil.copy(f, dst / sub)
shutil.copy(YOLO_DIR / 'stats.json', dst)
shutil.copy(YOLO_DIR / 'data.yaml', dst)
!cd {OUT_DIR} && rm -f results.zip && zip -qr results.zip . -x results.zip && ls -la {OUT_DIR} {OUT_DIR}/yolov8
r = json.load(open(dst / 'test_metrics.json'))
print(f"test: P={r['precision']:.4f} R={r['recall']:.4f} F1={r['f1']:.4f} mAP50={r['mAP50']:.4f} "
      f"(gốc: 0.9947 / 0.9957 / 0.9952 / 0.9947)")

**Dùng ở repo:** tải `output/results.zip`, đặt `yolov8/yolo_vaipe.pt` → `code/weights/yolo_vaipe.pt`;
`test_metrics.json` và các hình (`test/confusion_matrix*.png`, `results.png`) → `results/kaggle_yolov8/` cho báo cáo.